In [32]:
# pip install pymssql


In [1]:
import pandas as pd
import numpy as np
import pymssql
from datetime import datetime
from dateutil.relativedelta import relativedelta
from pandas import json_normalize
# import requests

import warnings
warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 500)
pd.set_option('display.width', 1000) # Optional, to prevent wrapping
# pd.set_option('display.float_format', lambda x: '%.3f' % x)

# %% [markdown]
# **SQL Queries:**

# %%
# Establish parameters for connection: 
conn = pymssql.connect(
    server='SQL2022CL02',
    database='JDE_PRODUCTION'
)  


In [ ]:
# 
# WMBECode_SQL_QUERY = """
# SELECT  [DRKY] as WMBE_Code_acronym
#       ,[DRDL01] as WMBE_Code_Name
#   FROM [JDE_PRODUCTION].[PRODCTL].[F0005]
#       WHERE DRSY = '01' AND LTRIM(RTRIM(DRKY)) in ('FD','MO','SBO','VO','WO')
# """

CatCode_SQL_QUERY = """
SELECT distinct LTRIM(rtrim(DRKY)) as DRKY_Code, [DRDL01] as Desciption, LTRIM(rtrim(DRKY)) + ' - ' + [DRDL01] AS CatCodeDesc
From [JDE_PRODUCTION].[PRODCTL].[F0005]
    WHERE DRSY = '41' AND DRRT = 'P5'
"""

Transaction_SQL_Query = """
WITH 
-- extract classfication code from Address Table F0101
ClassCode as (select [ABAN8], [ABCLASS01], [ABCLASS02], [ABCLASS03],[ABCLASS04], [ABCLASS05]
FROM [JDE_PRODUCTION].[PRODDTA].[F0101]
)

--,
-- CatCode as (SELECT distinct LTRIM(rtrim(DRKY)) as DRKY, [DRDL01], LTRIM(rtrim(DRKY)) + ' - ' + [DRDL01] AS CatCodeDesc
-- From [JDE_PRODUCTION].[PRODCTL].[F0005]
--    WHERE DRSY = '41' AND DRRT = 'P5'
--   --Where DRSY = '40' AND DRRT = 'AS'
-- )

SELECT
APLedger.[RPAN8], 
APLedger.[RPDCT], 
APLedger.[RPDOC],
[ABALPH],           
LTRIM(rtrim(PHPRP5)) AS PHPRP5,
--[CatCodeDesc],
[RNPAAP],  
(CAST(APDetail.[RNPAAP] as float) / 100)* -1 as 'Payment Amount',
[ABCLASS01], [ABCLASS02], [ABCLASS03],[ABCLASS04], [ABCLASS05],
 DATEADD(DAY, (APLedger.[RPDGJ] % 1000) - 1, 
              DATEFROMPARTS( (APLedger.[RPDGJ] / 1000) + 1900, 1, 1)) AS 'Date - For G/L (and Voucher)'
              
-- PHPRP5 AS Purchasing_Cat_Code_5
FROM [JDE_PRODUCTION].[PRODDTA].[F0411] as APLedger
INNER JOIN [JDE_PRODUCTION].[PRODDTA].[F0414] as APDetail
-- Join info from Design note
-- SFXE: Pay item extention number/ SFX: Document Pay Item / KCO:Document Company	/ DCT:Document Type / DOC:Document (Voucher Invoice etc.)	
  ON APLedger.[RPSFXE] = APDetail.[RNSFXE]
 AND APLedger.[RPSFX] = APDetail.[RNSFX]
 AND APLedger.[RPKCO] = APDetail.[RNKCO]
 AND APLedger.[RPDCT] = APDetail.[RNDCT]
 AND APLedger.[RPDOC] = APDetail.[RNDOC]

INNER JOIN [JDE_PRODUCTION].[PRODDTA].[F4301] as PHeader
-- 'Purchasing Category code 5' from Design notes shows below join conditions:
-- RPPO: Purchase Order /RPPDCT: Document Type - Purchase Order /RPPKCO: Document Company	
-- PHDOCO:Document (Order No Invoice etc.)	/PHDCTO: Order Type	/ PHKCOO: Order Company (Order Number)	
    ON  APLedger.[RPPO]   = PHeader.[PHDOCO]
    AND APLedger.[RPPDCT] = PHeader.[PHDCTO]
    AND APLedger.[RPPKCO] = PHeader.[PHKCOO]

LEFT JOIN ClassCode 
    ON APLedger.[RPAN8] = ClassCode.[ABAN8]

-- LEFT JOIN CatCode
--    ON PHeader.[PHPRP5] = CatCode.DRKY


WHERE APLedger.RPDCT IN ('P2','P3','PN','PP','PS','PR','PV')
  AND NULLIF(LTRIM(RTRIM(PHeader.[PHPRP5])), '') IS NOT NULL
  AND PHeader.[PHPRP5] NOT IN ('GRX','GRV','INT','MOU','LS','HS','OT','PA','PP','US')
  -- AND APLedger.RPDGJ <= '2023-12-31'
  AND APLedger.[RPFY] BETWEEN 18 AND 23
  -- AND APLedger.[RPFY] = 18

-- AND ltrim(PHeader.[PHPRP5]) = 'JOC'
-- AND (
  -- NULLIF(LTRIM(RTRIM([ABCLASS01])), '') IS NOT NULL
  --  OR NULLIF(LTRIM(RTRIM([ABCLASS02])), '') IS NOT NULL
  --  OR 
  -- NULLIF(LTRIM(RTRIM([ABCLASS03])), '') IS NOT NULL
  --  OR NULLIF(LTRIM(RTRIM([ABCLASS04])), '') IS NOT NULL
  --  OR
  --  AND NULLIF(LTRIM(RTRIM([ABCLASS05])), '') IS NOT NULL
  --  )
  
"""

# Load the queries: 

# df_WMBEcode = pd.read_sql_query(WMBECode_SQL_QUERY, conn)
df_transaction = pd.read_sql_query(Transaction_SQL_Query,conn)
# df_CatCode = pd.read_sql_query(CatCode_SQL_QUERY, conn)

#Close the connection:

conn.close()

In [3]:
df_WMBEcode

,WMBE_Code_acronym,WMBE_Code_Name
0,FD,Federally Disadvantage
1,MO,Minority Owned
2,SBO,Small Business Owned
3,VO,Veteran Owned
4,WO,Woman Owned


In [4]:
df_transaction

,PHPRP5,RNPAAP,Payment Amount,ABCLASS01,ABCLASS02,ABCLASS03,ABCLASS04,ABCLASS05,Date - For G/L (and Voucher)
0,MAT,-7480.0,74.80,,,,,,2018-01-18
1,MAT,-7480.0,74.80,,,,,,2018-01-18
2,MAT,-7480.0,74.80,,,,,,2018-01-18
3,MAT,-26180.0,261.80,,,,,,2018-01-18
4,MAT,-26180.0,261.80,,,,,,2018-01-18
...,...,...,...,...,...,...,...,...,...
91944,PST,-34347.0,343.47,,,,,,2023-12-31
91945,AEP,-40594.0,405.94,,,,,,2023-12-15
91946,PSP,-1570125.0,15701.25,,,,,,2023-12-31
91947,MAT,-54698.0,546.98,,,,,,2023-12-31


### df_CatCode: Dim Table main category list (adding Purchasing/Pw/Consulting)


In [37]:
df_CatCode

,DRKY_Code,Desciption,CatCodeDesc
0,,Blank - Landed Cost Rule 41/P5,- Blank - Landed Cost Rule 41/P5
1,1,Civil Engineering,1 - Civil Engineering
2,10,Environmental,10 - Environmental
3,100,Underground Utility Locates,100 - Underground Utility Locates
4,101,Traffic Control & Traffic Circ,101 - Traffic Control & Traffic Circ
...,...,...,...
868,ST,Software Telephone Support,ST - Software Telephone Support
869,STU,Structural Use,STU - Structural Use
870,SU,Software Upgrades,SU - Software Upgrades
871,SW,Small Public Work,SW - Small Public Work


In [ ]:
# Define mapping using lists of codes per category
Public_works_codes = ['551', '577', 'JOC', 'PW', 'SW', '264', '874']
Consulting_codes = ['AEP','AET','PS','PSP','PST','AE','SD','325','35','42','49','864', '1', '10', '130', '163','2', '328', '36', '431', '441', '70', '9', '983']
Purchasing_codes = ['MAT', 'RS', 'SLP', 'SM', 'GS', 'A16', 'LM', 'BM', 'PE', 'PG', 'ST', '271', '342', '861', '908', '974', 'A01', '913', '947', 'A05', 'SU']

# Create a single mapping dict
code_to_category = {}

for code in Purchasing_codes:
    code_to_category[code] = 'Purchasing'

for code in Consulting_codes:
    code_to_category[code] = 'Consulting'

for code in Public_works_codes:
    code_to_category[code] = 'Public Works'

# Then map
df_CatCode['MainCategory'] = df_CatCode['DRKY_Code'].map(code_to_category).fillna('Other_To verify')
df_CatCode

,DRKY_Code,Desciption,CatCodeDesc,MainCategory
0,,Blank - Landed Cost Rule 41/P5,- Blank - Landed Cost Rule 41/P5,Other_To verify
1,1,Civil Engineering,1 - Civil Engineering,Other_To verify
2,10,Environmental,10 - Environmental,Other_To verify
3,100,Underground Utility Locates,100 - Underground Utility Locates,Other_To verify
4,101,Traffic Control & Traffic Circ,101 - Traffic Control & Traffic Circ,Other_To verify
...,...,...,...,...
868,ST,Software Telephone Support,ST - Software Telephone Support,Purchasing
869,STU,Structural Use,STU - Structural Use,Other_To verify
870,SU,Software Upgrades,SU - Software Upgrades,Other_To verify
871,SW,Small Public Work,SW - Small Public Work,Public Works


### df_transaction table

In [39]:
df_transaction.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 91949 entries, 0 to 91948
Data columns (total 9 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   PHPRP5                        91949 non-null  object 
 1   RNPAAP                        91949 non-null  float64
 2   Payment Amount                91949 non-null  float64
 3   ABCLASS01                     91949 non-null  object 
 4   ABCLASS02                     91949 non-null  object 
 5   ABCLASS03                     91949 non-null  object 
 6   ABCLASS04                     91949 non-null  object 
 7   ABCLASS05                     91949 non-null  object 
 8   Date - For G/L (and Voucher)  91949 non-null  object 
dtypes: float64(2), object(7)
memory usage: 6.3+ MB


In [ ]:
df_transaction.rename(columns={  'Federally Disadvantage', 'Minority Owned', 'Small Business Owned', 'Veteran Owned', 'Woman Owned'
    "ABCLASS01": "FD",
    "ABCLASS02": "MO",
    "ABCLASS03": "SBO",
    "ABCLASS04": "VO",
    "ABCLASS05": "WO",
    "Date - For G/L (and Voucher)": "Date"
    }, inplace=True)
df_transaction.head(5)

,PHPRP5,RNPAAP,Payment Amount,FD,MO,SBO,VO,WO,Date
0,MAT,-7480.0,-74.80,,,,,,2018-01-18
1,MAT,-7480.0,-74.80,,,,,,2018-01-18
2,MAT,-11220.0,-112.20,,,,,,2018-01-18
3,MAT,-6059.0,-60.59,,,,,,2018-01-23
4,SLP,-168685.0,-1686.85,,,,,,2018-01-11


In [ ]:
# cast it to date
df_transaction["Date"] = pd.to_datetime(df_transaction["Date"]).dt.date


In [42]:
business_types = ['Federally Disadvantage', 'Minority Owned', 'Small Business Owned', 'Veteran Owned', 'Woman Owned']
business_acronyms = ['FD', 'MO', 'SBO', 'VO', 'WO']

In [ ]:
# value = df_transaction.loc[0, 'SBO']
# value

# value = df_transaction.loc[0, 'SBO']

# if value.startswith(" "):
#     print("Has leading spaces")

# if value.endswith(" "):
#     print("Has trailing spaces")

'   '

In [45]:
# Strip spaces for all ABCLASS columns
df_transaction[business_acronyms] = df_transaction[business_acronyms].apply(lambda x: x.str.strip())

df_transaction[business_acronyms] = df_transaction[business_acronyms].replace('', np.nan)

df_transaction[business_acronyms] = df_transaction[business_acronyms].fillna(0)

In [ ]:
# value = df_transaction.loc[0, 'SBO']
# value

0

In [47]:
df_transaction

,PHPRP5,RNPAAP,Payment Amount,FD,MO,SBO,VO,WO,Date
0,MAT,-7480.0,-74.80,0,0,0,0,0,2018-01-18
1,MAT,-7480.0,-74.80,0,0,0,0,0,2018-01-18
2,MAT,-11220.0,-112.20,0,0,0,0,0,2018-01-18
3,MAT,-6059.0,-60.59,0,0,0,0,0,2018-01-23
4,SLP,-168685.0,-1686.85,0,0,0,0,0,2018-01-11
...,...,...,...,...,...,...,...,...,...
91944,MAT,-51501.0,-515.01,0,0,0,0,0,2023-12-31
91945,MAT,-513.0,-5.13,0,0,0,0,0,2023-12-31
91946,AEP,-131700.0,-1317.00,0,0,0,0,0,2023-12-31
91947,PS,-1301621.0,-13016.21,0,0,0,0,0,2023-12-31


In [ ]:
# df_transaction.iloc[5719]

PHPRP5                            MAT
RNPAAP                        -3712.0
Payment Amount                 -37.12
FD                                  0
MO                                  0
SBO                                 0
VO                                  0
WO                                  0
Date              2018-05-03 00:00:00
Name: 5719, dtype: object

In [ ]:
# total = sum(df_transaction['Payment Amount'])
# total

-740232805.560007

In [50]:
df_transaction.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 91949 entries, 0 to 91948
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   PHPRP5          91949 non-null  object        
 1   RNPAAP          91949 non-null  float64       
 2   Payment Amount  91949 non-null  float64       
 3   FD              91949 non-null  object        
 4   MO              91949 non-null  object        
 5   SBO             91949 non-null  object        
 6   VO              91949 non-null  object        
 7   WO              91949 non-null  object        
 8   Date            91949 non-null  datetime64[ns]
dtypes: datetime64[ns](1), float64(2), object(6)
memory usage: 6.3+ MB


In [51]:
df_transaction.describe()

,RNPAAP,Payment Amount,Date
count,9.194900e+04,9.194900e+04,91949
mean,-8.050472e+05,-8.050472e+03,2020-11-18 20:31:19.146048256
min,-3.184655e+08,-3.184655e+06,2018-01-02 00:00:00
25%,-3.190000e+05,-3.190000e+03,2019-05-02 00:00:00
50%,-6.826000e+04,-6.826000e+02,2020-10-05 00:00:00
75%,-1.650000e+04,-1.650000e+02,2022-06-09 00:00:00
max,2.457338e+08,2.457338e+06,2023-12-31 00:00:00
std,5.069039e+06,5.069039e+04,NaN


In [52]:
# replace, say SBO with 1
for i in range(len(business_acronyms)):
  ba = business_acronyms[i]
  # ba = business_acronyms[i]
  df_transaction[ba] = df_transaction[ba].replace(ba, 1)

df_transaction

,PHPRP5,RNPAAP,Payment Amount,FD,MO,SBO,VO,WO,Date
0,MAT,-7480.0,-74.80,0,0,0,0,0,2018-01-18
1,MAT,-7480.0,-74.80,0,0,0,0,0,2018-01-18
2,MAT,-11220.0,-112.20,0,0,0,0,0,2018-01-18
3,MAT,-6059.0,-60.59,0,0,0,0,0,2018-01-23
4,SLP,-168685.0,-1686.85,0,0,0,0,0,2018-01-11
...,...,...,...,...,...,...,...,...,...
91944,MAT,-51501.0,-515.01,0,0,0,0,0,2023-12-31
91945,MAT,-513.0,-5.13,0,0,0,0,0,2023-12-31
91946,AEP,-131700.0,-1317.00,0,0,0,0,0,2023-12-31
91947,PS,-1301621.0,-13016.21,0,0,0,0,0,2023-12-31


In [53]:
#  Create each transaction ids
df_transaction.insert(0, "Tran_id", df_transaction.index+1)

In [54]:
df_transaction

,Tran_id,PHPRP5,RNPAAP,Payment Amount,FD,MO,SBO,VO,WO,Date
0,1,MAT,-7480.0,-74.80,0,0,0,0,0,2018-01-18
1,2,MAT,-7480.0,-74.80,0,0,0,0,0,2018-01-18
2,3,MAT,-11220.0,-112.20,0,0,0,0,0,2018-01-18
3,4,MAT,-6059.0,-60.59,0,0,0,0,0,2018-01-23
4,5,SLP,-168685.0,-1686.85,0,0,0,0,0,2018-01-11
...,...,...,...,...,...,...,...,...,...,...
91944,91945,MAT,-51501.0,-515.01,0,0,0,0,0,2023-12-31
91945,91946,MAT,-513.0,-5.13,0,0,0,0,0,2023-12-31
91946,91947,AEP,-131700.0,-1317.00,0,0,0,0,0,2023-12-31
91947,91948,PS,-1301621.0,-13016.21,0,0,0,0,0,2023-12-31


### df_DimWMBEcode table

In [55]:
#create WMBE_code table  (unique wMBE types including "non-WMBE")

df_WMBEcode.reset_index(drop=True)
df_WMBEcode.insert(0, "WMBE_Code_num", df_WMBEcode.index+1)
df_WMBEcode

non_wmbe = pd.DataFrame({
    "WMBE_Code_num": [len(df_WMBEcode)+1],
    "WMBE_Code_acronym"	:["NonWMBE"],
    "WMBE_Code_Name": ["Non WMBE"]
})


df_DimWMBEcode=pd.concat([df_WMBEcode,non_wmbe], ignore_index=True)
df_DimWMBEcode

,WMBE_Code_num,WMBE_Code_acronym,WMBE_Code_Name
0,1,FD,Federally Disadvantage
1,2,MO,Minority Owned
2,3,SBO,Small Business Owned
3,4,VO,Veteran Owned
4,5,WO,Woman Owned
5,6,NonWMBE,Non WMBE


### df_transaction_wmbe mapping table ????

In [56]:
# # Create purchase_df (purchase details only)   tbc???
# purchase_df = df_transaction[['year', 'amount', 'cat_id']].reset_index(drop=True)
# purchase_df['p_id'] = purchase_df.index + 1  # Create unique purchase ids
# purchase_df = purchase_df[['p_id', 'cat_id', 'year', 'amount']]


 ### Fact: df_tran_wmbe_mapping


In [ ]:
# Create tran_wmbe_mapping (if one transaction has multiple wmbes, will separate them into multiple rows)
tran_wmbe_mapping = []

# Iterate through each purchase and identify the bu_codes
for i, row in df_transaction.iterrows():
    tran_id = i + 1  # Use row index for tran_id
    wmbe_ids = []

    for i in range(len(business_acronyms)):
        business_type = business_acronyms[i]
        if row[business_type] == 1:
            wmbe_ids.append(i + 1)  # Add the corresponding wmbe_id

    # If no wmbe_codes are 1, assign to "non-WMBE"
    if not wmbe_ids:
        wmbe_ids.append(6)

    # Create mappings
    for wmbe_id in wmbe_ids:
        tran_wmbe_mapping.append({'Tran_id': tran_id, 'WMBE_id': wmbe_id})

df_tran_wmbe_mapping = pd.DataFrame(tran_wmbe_mapping)
df_tran_wmbe_mapping

,Tran_id,WMBE_id
0,1,6
1,2,6
2,3,6
3,4,6
4,5,6
...,...,...
97443,91948,6
97444,91949,1
97445,91949,2
97446,91949,3


### df_transaction: Final revision

In [58]:
df_transaction

,Tran_id,PHPRP5,RNPAAP,Payment Amount,FD,MO,SBO,VO,WO,Date
0,1,MAT,-7480.0,-74.80,0,0,0,0,0,2018-01-18
1,2,MAT,-7480.0,-74.80,0,0,0,0,0,2018-01-18
2,3,MAT,-11220.0,-112.20,0,0,0,0,0,2018-01-18
3,4,MAT,-6059.0,-60.59,0,0,0,0,0,2018-01-23
4,5,SLP,-168685.0,-1686.85,0,0,0,0,0,2018-01-11
...,...,...,...,...,...,...,...,...,...,...
91944,91945,MAT,-51501.0,-515.01,0,0,0,0,0,2023-12-31
91945,91946,MAT,-513.0,-5.13,0,0,0,0,0,2023-12-31
91946,91947,AEP,-131700.0,-1317.00,0,0,0,0,0,2023-12-31
91947,91948,PS,-1301621.0,-13016.21,0,0,0,0,0,2023-12-31


In [ ]:
# #Spend: turn negative to positive /Credit: turn positve to negative
# df_transaction["PaymentAmount_Adjusted"] = df_transaction["Payment Amount"] * -1
# df_transaction


,Tran_id,PHPRP5,RNPAAP,Payment Amount,FD,MO,SBO,VO,WO,Date,PaymentAmount_Adjusted
0,1,MAT,-7480.0,-74.80,0,0,0,0,0,2018-01-18,74.80
1,2,MAT,-7480.0,-74.80,0,0,0,0,0,2018-01-18,74.80
2,3,MAT,-11220.0,-112.20,0,0,0,0,0,2018-01-18,112.20
3,4,MAT,-6059.0,-60.59,0,0,0,0,0,2018-01-23,60.59
4,5,SLP,-168685.0,-1686.85,0,0,0,0,0,2018-01-11,1686.85
...,...,...,...,...,...,...,...,...,...,...,...
91944,91945,MAT,-51501.0,-515.01,0,0,0,0,0,2023-12-31,515.01
91945,91946,MAT,-513.0,-5.13,0,0,0,0,0,2023-12-31,5.13
91946,91947,AEP,-131700.0,-1317.00,0,0,0,0,0,2023-12-31,1317.00
91947,91948,PS,-1301621.0,-13016.21,0,0,0,0,0,2023-12-31,13016.21


#### (remove WMBE section since we have df_tran_wmbe_mapping)

In [ ]:
df_transaction.drop(['RNPAAP', *business_acronyms], axis=1, inplace=True)
df_transaction


,Tran_id,PHPRP5,Date,PaymentAmount_Adjusted
0,1,MAT,2018-01-18,74.80
1,2,MAT,2018-01-18,74.80
2,3,MAT,2018-01-18,112.20
3,4,MAT,2018-01-23,60.59
4,5,SLP,2018-01-11,1686.85
...,...,...,...,...
91944,91945,MAT,2023-12-31,515.01
91945,91946,MAT,2023-12-31,5.13
91946,91947,AEP,2023-12-31,1317.00
91947,91948,PS,2023-12-31,13016.21


In [61]:
# # filter out transactions belongs to any WMBE type for verification
# df_transaction[df_transaction[business_acronyms].eq(1).any(axis=1)]

# df_transaction[df_transaction[business_acronyms].eq(0).all(axis=1)]


### df_DimDate table

In [63]:
# Create a date dimension table
df_DimDate = pd.DataFrame({"Date": pd.date_range("2015-01-01", "2030-12-31")})
df_DimDate["Year"] = df_DimDate["Date"].dt.year
df_DimDate["Month Number"] = df_DimDate["Date"].dt.month
df_DimDate["Month Name"] = df_DimDate["Date"].dt.strftime("%B")
df_DimDate["YearMonth"] = df_DimDate["Date"].dt.strftime("%Y-%m")
df_DimDate["Quarter"] = df_DimDate["Date"].dt.to_period("Q").astype(str)


In [64]:
df_DimDate

,Date,Year,Month Number,Month Name,YearMonth,Quarter
0,2015-01-01,2015,1,January,2015-01,2015Q1
1,2015-01-02,2015,1,January,2015-01,2015Q1
2,2015-01-03,2015,1,January,2015-01,2015Q1
3,2015-01-04,2015,1,January,2015-01,2015Q1
4,2015-01-05,2015,1,January,2015-01,2015Q1
...,...,...,...,...,...,...
5839,2030-12-27,2030,12,December,2030-12,2030Q4
5840,2030-12-28,2030,12,December,2030-12,2030Q4
5841,2030-12-29,2030,12,December,2030-12,2030Q4
5842,2030-12-30,2030,12,December,2030-12,2030Q4


### Final scripts import to PowerBI

In [ ]:
import pandas as pd
import numpy as np
import pymssql
from datetime import datetime

# Establish parameters for connection: 
conn = pymssql.connect(
    server='SQL2022CL02',
    database='JDE_PRODUCTION'
)  
# 
# WMBECode_SQL_QUERY = """
# SELECT  [DRKY] as WMBE_Code_acronym
#       ,[DRDL01] as WMBE_Code_Name
#   FROM [JDE_PRODUCTION].[PRODCTL].[F0005]
#       WHERE DRSY = '01' AND LTRIM(RTRIM(DRKY)) in ('FD','MO','SBO','VO','WO')
# """

CatCode_SQL_QUERY = """
SELECT distinct LTRIM(rtrim(DRKY)) as DRKY_Code, [DRDL01] as Desciption, LTRIM(rtrim(DRKY)) + ' - ' + [DRDL01] AS CatCodeDesc
From [JDE_PRODUCTION].[PRODCTL].[F0005]
    WHERE DRSY = '41' AND DRRT = 'P5'
"""

Transaction_SQL_Query = """
WITH 
-- extract classfication code from Address Table F0101
ClassCode as (select [ABAN8], [ABCLASS01], [ABCLASS02], [ABCLASS03],[ABCLASS04], [ABCLASS05]
FROM [JDE_PRODUCTION].[PRODDTA].[F0101]
)

--,
-- CatCode as (SELECT distinct LTRIM(rtrim(DRKY)) as DRKY, [DRDL01], LTRIM(rtrim(DRKY)) + ' - ' + [DRDL01] AS CatCodeDesc
-- From [JDE_PRODUCTION].[PRODCTL].[F0005]
--    WHERE DRSY = '41' AND DRRT = 'P5'
--   --Where DRSY = '40' AND DRRT = 'AS'
-- )
   
SELECT  
CAST(PHDOCO AS INT) AS PONumber,
LTRIM(rtrim(PHPRP5)) AS PHPRP5,
--[CatCodeDesc],
[RNPAAP],  
(CAST(APDetail.[RNPAAP] as float) / 100)* -1 as 'Payment Amount',
[ABCLASS01],
[ABCLASS02], 
[ABCLASS03],
[ABCLASS04], 
[ABCLASS05],
[ABALPH], 
DATEADD(DAY, (APLedger.[RPDGJ] % 1000) - 1, 
              DATEFROMPARTS( (APLedger.[RPDGJ] / 1000) + 1900, 1, 1)) AS 'Date - For G/L (and Voucher)'
              
-- PHPRP5 AS Purchasing_Cat_Code_5
FROM [JDE_PRODUCTION].[PRODDTA].[F0411] as APLedger
INNER JOIN [JDE_PRODUCTION].[PRODDTA].[F0414] as APDetail
-- Join info from Design note
-- SFXE: Pay item extention number/ SFX: Document Pay Item / KCO:Document Company	/ DCT:Document Type / DOC:Document (Voucher Invoice etc.)	
  ON APLedger.[RPSFXE] = APDetail.[RNSFXE]
 AND APLedger.[RPSFX] = APDetail.[RNSFX]
 AND APLedger.[RPKCO] = APDetail.[RNKCO]
 AND APLedger.[RPDCT] = APDetail.[RNDCT]
 AND APLedger.[RPDOC] = APDetail.[RNDOC]

INNER JOIN [JDE_PRODUCTION].[PRODDTA].[F4301] as PHeader
-- 'Purchasing Category code 5' from Design notes shows below join conditions:
-- RPPO: Purchase Order /RPPDCT: Document Type - Purchase Order /RPPKCO: Document Company	
-- PHDOCO:Document (Order No Invoice etc.)	/PHDCTO: Order Type	/ PHKCOO: Order Company (Order Number)	
    ON  APLedger.[RPPO]   = PHeader.[PHDOCO]
    AND APLedger.[RPPDCT] = PHeader.[PHDCTO]
    AND APLedger.[RPPKCO] = PHeader.[PHKCOO]

LEFT JOIN ClassCode 
    ON APLedger.[RPAN8] = ClassCode.[ABAN8]

-- LEFT JOIN CatCode
--    ON PHeader.[PHPRP5] = CatCode.DRKY


WHERE APLedger.RPDCT IN ('P2','P3','PN','PP','PS','PR','PV')
  AND NULLIF(LTRIM(RTRIM(PHeader.[PHPRP5])), '') IS NOT NULL
  AND PHeader.[PHPRP5] NOT IN ('GRX','GRV','INT','MOU','LS','HS','OT','PA','PP','US')
  AND APLedger.RPDGJ >= ((2018 - 1900) * 1000 + DATEPART(DAYOFYEAR, '2018-01-01'))
  AND APLedger.RPDGJ <= (YEAR(GETDATE()) - 1900) * 1000 + DATEPART(DAYOFYEAR, DATEADD(DAY, -1, GETDATE()))

"""

# Load the queries: 

df_WMBEcode = pd.read_sql_query(WMBECode_SQL_QUERY, conn)
df_transaction = pd.read_sql_query(Transaction_SQL_Query,conn)
df_CatCode = pd.read_sql_query(CatCode_SQL_QUERY, conn)

#Close the connection:

conn.close()

#  Create each transaction ids
df_transaction.insert(0, "Tran_id", df_transaction.index+1)
df_transaction["PONumber"]=df_transaction["PONumber"].astype("Int64")


df_transaction.rename(columns={
    "ABCLASS01": "FD",
    "ABCLASS02": "MO",
    "ABCLASS03": "SBO",
    "ABCLASS04": "VO",
    "ABCLASS05": "WO",
    "Date - For G/L (and Voucher)": "Date"
    }, inplace=True)

df_transaction["Date"] = pd.to_datetime(df_transaction["Date"]).dt.date
df_transaction['RPDOC'] = df_transaction['RPDOC'].astype(int)


business_types = ['Federally Disadvantage', 'Minority Owned', 'Small Business Owned', 'Veteran Owned', 'Woman Owned']
business_acronyms = ['FD', 'MO', 'SBO', 'VO', 'WO']

# Strip spaces for all ABCLASS columns
df_transaction[business_acronyms] = df_transaction[business_acronyms].apply(lambda x: x.str.strip())

df_transaction[business_acronyms] = df_transaction[business_acronyms].replace('', np.nan)

df_transaction[business_acronyms] = df_transaction[business_acronyms].fillna(0)



# replace, say SBO with 1
for i in range(len(business_acronyms)):
  ba = business_acronyms[i]
  # ba = business_acronyms[i]
  df_transaction[ba] = df_transaction[ba].replace(ba, 1)

# add "WMBE Tpype" column for table use in PowerBi
wmbe_columns = {
    "FD": "FD",
    "MO": "MO",
    "SBO": "SBO",
    "VO": "VO",
    "WO": "WO"
}

# Create the WMBE Types column
df_transaction["WMBE Types"] = df_transaction.apply(
    lambda row: ", ".join([label for col, label in wmbe_columns.items() if row[col] == 1]) 
    if any(row[col] == 1 for col in wmbe_columns) 
    else "Non-WMBE",
    axis=1
)



# Create tran_wmbe_mapping (if one transaction has multiple wmbes, will separate them into multiple rows)
tran_wmbe_mapping = []

# Iterate through each purchase and identify the bu_codes
for i, row in df_transaction.iterrows():
    tran_id = i + 1  # Use row index for tran_id
    wmbe_ids = []

    for i in range(len(business_acronyms)):
        business_type = business_acronyms[i]
        if row[business_type] == 1:
            wmbe_ids.append(i + 1)  # Add the corresponding wmbe_id

    # If no wmbe_codes are 1, assign to "non-WMBE"
    if not wmbe_ids:
        wmbe_ids.append(6)

    # Create mappings
    for wmbe_id in wmbe_ids:
        tran_wmbe_mapping.append({'Tran_id': tran_id, 'WMBE_id': wmbe_id})

df_tran_wmbe_mapping = pd.DataFrame(tran_wmbe_mapping)
df_tran_wmbe_mapping

df_transaction.drop(['RNPAAP', *business_acronyms], axis=1, inplace=True)
df_transaction


# Define mapping using lists of codes per category
Public_works_codes = ['551', '577', 'JOC', 'PW', 'SW', '264', '874']
Consulting_codes = ['AEP','AET','PS','PSP','PST','AE','SD','325','35','42','49','864', '1', '10', '130', '163','2', '328', '36', '431', '441', '70', '9', '983']
Purchasing_codes = ['MAT', 'RS', 'SLP', 'SM', 'GS', 'A16', 'LM', 'BM', 'PE', 'PG', 'ST', '271', '342', '861', '908', '974', 'A01', '913', '947', 'A05', 'SU']

# Create a single mapping dict
code_to_category = {}

for code in Purchasing_codes:
    code_to_category[code] = 'Purchasing'

for code in Consulting_codes:
    code_to_category[code] = 'Consulting'

for code in Public_works_codes:
    code_to_category[code] = 'Public Works'

# Then map
df_CatCode['MainCategory'] = df_CatCode['DRKY_Code'].map(code_to_category).fillna('Other_To verify')
df_CatCode